# judge 프롬프트와 편향

Python 3.12 새 커널에서 후보 순서를 바꾸는 위치 편향 실험부터 확인합니다.

## Python 3.12 패키지 버전 고정

In [1]:
# 실습 환경에서 사용하는 보조 패키지 버전을 고정합니다.
%pip install -q python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 후보 순서를 바꾸는 위치 편향 실험

In [2]:
# 첫 후보만 고르는 잘못된 판정기로 위치 편향이 생기는 구조를 재현합니다.
short = ('short', '도서는 7권까지 14일 동안 빌릴 수 있습니다. [E01]')
long = ('long', short[1] + ' 도서관은 시민을 위한 지식 공간입니다.')
def biased_pairwise(first: tuple[str, str], second: tuple[str, str]) -> str:
    return first[0]  # 잘못된 평가자: 내용 대신 첫 후보 ID를 선택
winner_ab = biased_pairwise(short, long)
winner_ba = biased_pairwise(long, short)
print('short/long 승자:', winner_ab)
print('long/short 승자:', winner_ba)
print('순서를 바꿔도 같은 후보인가:', winner_ab == winner_ba)

short/long 승자: short
long/short 승자: long
순서를 바꿔도 같은 후보인가: False


## 평가 모델 사양과 프롬프트 해시 고정

In [3]:
# judge 모델·온도·프롬프트 해시·출력 형식을 하나의 사양으로 묶습니다.
import hashlib
judge_prompt = '근거로 뒷받침되면 1, 아니면 0. JSON만 출력.'
spec = {
 'model': 'gpt-5.4-mini', 'temperature': 0,
 'prompt_version': 'groundedness-v1.0.0',
 'prompt_sha256': hashlib.sha256(judge_prompt.encode()).hexdigest(),
 'output_schema': {'score': '0|1', 'reason': 'string'},
}
print(spec)

{'model': 'gpt-5.4-mini', 'temperature': 0, 'prompt_version': 'groundedness-v1.0.0', 'prompt_sha256': 'e8b76f93d743595c23e7812d6d5b26a85b4bc1ee407dfa89313850d0261793e7', 'output_schema': {'score': '0|1', 'reason': 'string'}}


## 저장된 실제 평가 모델 버전 확인

In [4]:
# 세 사례의 실제 judge 기록과 운영 모델 미정 상태를 확인합니다.
from pathlib import Path
import json
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'labs/part4/ch15-evaluation-advanced').is_dir())
data = json.loads((root / 'labs/part4/ch15-evaluation-advanced/data/evaluation-cases.json').read_text())
for case in data['cases']:
    print(case['qid'], case['judge_model'], case['judge_prompt_version'])
assert all(x['judge_placeholder'] == '[[미정: judge 모델]]' for x in data['cases'])

Q002 gpt-5.4-mini groundedness-v1.0.0
Q006 gpt-5.4-mini groundedness-v1.0.0
Q090 gpt-5.4-mini groundedness-v1.0.0
